<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-28</br>
</div>

</br>

In [ ]:
# TODO 0: 환경변수를 로드하고, LLM, 검색기, Agent를 준비해봅시다.

import os, warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

# LLM 초기화
from langchain_upstage import ChatUpstage, UpstageEmbeddings

llm = ChatUpstage(model="solar-pro3", temperature=0)

# 정책 문서 정의
from langchain_core.documents import Document

refund_policy = """# 환불 정책 (Refund Policy)

## 환불 조건
1. 구매 후 7일 이내: 무조건 환불 가능 (환불 승인)
2. 구매 후 7일 초과: 환불 불가 (환불 거부)
3. 상품 불량 시: 30일 이내 환불 가능

## 환불 절차
- 환불 요청 접수 → 상태 확인 → 승인/거부 → 처리 완료
- 승인 시 3~5 영업일 내 환불 처리

## 주의사항
- 사용 흔적이 있는 상품은 환불 불가
- 디지털 상품은 다운로드 후 환불 불가"""

reward_policy = """# 보상 정책 (Reward Policy)

## 쿠폰 발급 조건
1. 배송 지연 시: 최대 10,000원 쿠폰 발급 가능
2. 상품 불량 시: 최대 20,000원 쿠폰 발급 가능
3. 단순 변심: 쿠폰 발급 불가

## 쿠폰 발급 한도
- 1회 최대 발급 금액: 20,000원
- 월간 누적 한도: 50,000원

## 주의사항
- 배송 완료 후 7일 이내에만 보상 가능
- 이미 쿠폰을 받은 주문에는 추가 발급 불가"""

policy_documents = [
    Document(page_content=refund_policy, metadata={"source": "환불 정책"}),
    Document(page_content=reward_policy, metadata={"source": "보상 정책"}),
]

# 청킹 → 임베딩 → 벡터스토어 → 검색기
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
split_docs = text_splitter.split_documents(policy_documents)

embeddings = UpstageEmbeddings(model="embedding-query")
vectorstore = Chroma.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# search_policy 도구 정의 (001에서 학습)
from langchain_core.tools import tool

@tool
def search_policy(query: str) -> str:
    """고객 서비스 정책을 검색합니다."""
    documents = retriever.invoke(query)
    return "\n".join([doc.page_content for doc in documents])

tools = [search_policy]

# ReAct Agent 생성 (001에서 학습)
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

system_prompt = "당신은 고객 서비스 AI 에이전트입니다. 고객의 요청을 처리하기 위해 제공된 도구를 사용하세요. 환불 정책: 구매 후 7일 이내 환불 승인, 7일 초과 환불 거부. 최종 결론을 먼저 말하고, 근거를 설명하세요."

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=MemorySaver()
)

print(f"✅ 환경 설정 완료 (정책 문서 {len(split_docs)}개 청크, Agent 생성)")

</br>

# 학습 내용
>이번 장에서는 <strong>Trustworthiness(신뢰성 확보)</strong>에 대해 학습합니다.
>Agent가 안전하고 정확하게 동작하는지 검증하는 평가 체계와 가드레일을 학습해봅시다.

<div style="text-align:center">

</div>

</br>

# Agent 신뢰성 (Trustworthiness)
> Agent가 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">의도한 대로 정확하게 동작</mark>하는지 체계적으로 검증하는 것입니다.

> Agent가 그럴듯하게 작동하는 것처럼 보여도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제로 올바른 결정을 내리고 있는지는 별개의 문제</mark>입니다.</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">할루시네이션(Hallucination)</mark>으로 LLM이 존재하지 않는 정책이나 사실을 자신감 있게 말할 수 있고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">잘못된 정보로 인한 피해</mark>로 "환불 가능합니다"라고 잘못 안내하면 고객 불만과 비즈니스 손실이 발생합니다.</br>
> 또한 프로덕션에 배포된 Agent가 예외 케이스에서 어떻게 동작하는지 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">AI 안전성</mark> 측면에서 사전에 검증해야 합니다.</br></br>
> 단순히 "자연스럽게 답하는가"가 아니라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비즈니스 규칙에 정확히 부합하는가</mark>를 체계적으로 검증하는 전략이 필요합니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">신뢰성 요소</th>
      <th>설명</th>
      <th>예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">정확성</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">올바른 정보 제공</mark></td><td>정책에 맞는 환불 안내</td></tr>
    <tr><td style="text-align:center">안전성</td><td>위험한 행동 방지</td><td>무분별한 환불 승인 차단</td></tr>
    <tr><td style="text-align:center">일관성</td><td>동일 입력 → 동일 출력</td><td>같은 질문에 같은 답변</td></tr>
  </tbody>
</table>

</br>

## 환불 상태 추출 함수
> Agent 응답에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">환불 승인/거부 상태</mark>를 정규식으로 추출합니다.

In [ ]:
# TODO 1: 환불 상태 추출 함수를 정의하세요. 정규식으로 승인/거부 패턴의 마지막 등장 위치를 비교하여 최종 결론을 추출하세요. 3개의 테스트 응답으로 결과를 확인하세요.

import re

# TODO 1-1: extract_refund_statuses 함수를 정의하세요.

    # 매칭된 패턴이 하나뿐이면 그대로 반환
    # 둘 다 매칭되면 마지막 등장 위치가 결론

# TODO 1-2: 3개의 테스트 응답으로 결과를 확인하세요.

</br>

## Agent 평가 함수
> 여러 테스트 케이스에 대해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기대 결과와 실제 결과를 비교</mark>합니다.

In [ ]:
# TODO 2: Agent 평가 함수를 정의하세요. 테스트 케이스를 순회하며 Agent 실행 결과에서 환불 상태를 추출하고, 기대 결과와 비교하여 PASS/FAIL을 출력한 뒤, 최종 정확도(correct_count/total_count)를 반환하세요.

# TODO 2-1: evaluate_agent 함수의 시그니처와 초기 변수를 정의하세요.

    # TODO 2-2: 각 테스트 케이스를 순회하며 PASS/FAIL을 출력하세요.

    # TODO 2-3: 정확도를 계산하여 출력하고 반환하세요.

</br>

## 테스트 케이스 설계

In [ ]:
# TODO 3: 테스트 케이스 리스트를 작성하세요. 각 케이스는 질문 문자열과 기대 결과(["approved"] 또는 ["denied"])로 구성합니다. 7일 이내는 승인, 초과는 거부입니다. 4개의 테스트 케이스를 만드세요.

In [ ]:
# TODO 4: 평가 함수로 Agent를 평가하세요.

💡평가의 핵심
> 단순히 "답변이 자연스러운가"가 아니라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">비즈니스 로직에 맞는가</mark>를 검증합니다.
> 환불 정책이 "7일 이내"라면, Agent도 정확히 그 기준을 따라야 합니다.

💡정규식 추출의 한계
> LLM 응답은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">형식이 일정하지 않을</mark> 수 있습니다.
> 구조화된 출력(JSON mode)을 사용하면 더 안정적인 평가가 가능합니다.

</br>

# 안전한 도구 패턴 (Safety Tool)
> 도구 실행 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전에 검증 로직</mark>을 넣어 정책 위반을 사전에 차단합니다.

> LLM은 도구를 호출할 때 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정책 한도를 초과하는 금액</mark>을 전달하거나, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">조건을 충족하지 않는 주문</mark>에 쿠폰을 발급할 수 있습니다.</br></br>
> 검증 없는 `issue_coupon` 함수는 어떤 금액이든 그대로 발급하므로 정책 위반이 발생합니다.</br></br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Safety Tool 패턴</mark>은 도구 함수 내부에 검증 로직을 포함하여, 정책 위반 요청을 실행 전에 거부하는 방식입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교</th>
      <th>issue_coupon (기본)</th>
      <th>issue_coupon_safe (안전)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">금액 검증</td><td>없음</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정책 한도 확인</mark></td></tr>
    <tr><td style="text-align:center">주문 상태 확인</td><td>없음</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배송 지연만 허용</mark></td></tr>
    <tr><td style="text-align:center">정책 위반 시</td><td>그대로 실행</td><td>거부 메시지 반환</td></tr>
  </tbody>
</table>

In [ ]:
# TODO 5: 검증 로직이 포함된 issue_coupon_safe 함수를 정의하세요. 금액이 MAX_COUPON_AMOUNT(20000)을 초과하면 거부하고, 주문 상태가 "배송 지연"이 아니면 거부하세요.

    # 1. 금액 검증

    # 2. 주문 존재 확인

    # 3. 주문 상태 확인

    # 4. 모든 검증 통과 → 실행

In [ ]:
# TODO 6: issue_coupon_safe의 3가지 케이스를 테스트하세요. 한도 초과, 조건 미충족, 정상 발급 각각을 실행하여 결과를 확인하세요.

# 테스트 1: 한도 초과 (10만원 발급 시도)

# 테스트 2: 조건 미충족 (배송 완료 주문)

# 테스트 3: 정상 발급 (배송 지연 + 한도 내)

💡Safety Tool 패턴의 원칙
> 도구 함수 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">내부에 검증 로직</mark>을 포함하면, LLM이 어떤 인자를 전달하든 정책을 위반하는 실행은 차단됩니다.
> LLM의 판단에 의존하지 않고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">코드 레벨에서 강제</mark>하는 것이 핵심입니다.

</br>

# 가드레일 (Guardrail)
> Agent의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력과 출력을 필터링</mark>하여 유해한 요청 차단 및 민감 정보 노출을 방지합니다.

> 도구 내부 검증(Safety Tool)만으로는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">모든 위협을 막을 수 없습니다</mark>.</br></br>
> 사용자가 "시스템 접근 권한을 줘"와 같은 유해한 요청을 보내거나, Agent 응답에 주민등록번호 같은 민감 정보가 포함될 수 있습니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">가드레일</mark>은 Agent 실행 전후에 배치되어 입력/출력을 필터링하는 보호 계층입니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">가드레일 유형</th>
      <th>역할</th>
      <th>적용 위치</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">입력 가드레일</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">유해 요청 차단</mark> (해킹, 시스템 접근 등)</td><td>Agent 실행 전</td></tr>
    <tr><td style="text-align:center">출력 가드레일</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">민감 정보 필터링</mark> (주민번호, 계좌번호 등)</td><td>응답 반환 전</td></tr>
  </tbody>
</table>

## 입력 가드레일
> 사용자 입력에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">차단 키워드</mark>를 감지하여 유해한 요청을 사전에 거부합니다.

In [ ]:
# TODO 7: 입력 가드레일 함수를 정의하세요. 차단 키워드 리스트를 순회하며 입력에 포함되어 있으면 True를 반환하세요.

# TODO 7-1: is_harmful_input 함수를 정의하세요.

# TODO 7-2: 3개의 테스트 입력으로 결과를 확인하세요.

## 출력 가드레일
> Agent 응답에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">민감 정보를 마스킹</mark>하여 노출을 방지합니다.

In [ ]:
# TODO 8: 출력 가드레일 함수를 정의하세요. 민감 키워드와 번호 패턴을 "[민감정보 필터링됨]"으로 대체하세요.

import re

# TODO 8-1: filter_output 함수를 정의하세요.

# TODO 8-2: 3개의 테스트 출력으로 결과를 확인하세요.

</br>

## TrustedAgentExecutor 패턴
> 입력 가드레일, Agent 실행, 출력 가드레일을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">하나의 클래스로 통합</mark>합니다.

In [ ]:
# TODO 9: TrustedAgentExecutor 클래스를 정의하세요. run 메서드에서 입력 가드레일 → Agent 실행 → 출력 가드레일 순서로 처리하세요.

    # TODO 9-1: __init__ 메서드를 정의하여 agent를 저장하세요.

        # TODO 9-2: 입력 가드레일을 적용하세요.

        # TODO 9-3: Agent를 실행하세요.

        # TODO 9-4: 출력 가드레일을 적용하여 반환하세요.

In [ ]:
# TODO 10: TrustedAgentExecutor를 생성하고 정상 요청과 유해 요청을 각각 테스트하세요.

# 정상 요청

# 유해한 요청

💡Trustworthiness 보호 계층 정리
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">입력 가드레일 → Agent/ReAct → Safety Tool(검증 로직) → 출력 가드레일</mark> 순서로 보호 계층이 적용됩니다.
> 각 계층은 독립적으로 동작하므로, 하나의 계층이 놓친 위협을 다른 계층에서 잡을 수 있습니다.